# 065. next character 를 예측하여 Shakespeare Sonnet Text 생성

- 순환 신경망을 활용한 문자열 생성


- character 단위로 text 생성  

    - 문자 시퀀스 (ex. "Shakespear")가 주어지면, 시퀀스의 다음 문자("e")를 예측하는 모델을 훈련
    - 모델을 반복하여 호출하면 더 긴 텍스트 시퀀스 생성이 가능
    - 훈련이 시작되었을 때, 이 모델은 영어 단어의 철자를 모르거나 심지어 텍스트의 단위가 단어라는 것도 모름 
    
    
- google NLP tutorial 참조

In [1]:
import tensorflow as tf
import numpy as np
import time
print(tf.__version__)

2.0.0


### Text File download

- 셰익스피어 데이터셋 다운로드  


- data 는 ~/.keras/dataset 에 저장. absolute path 지정해 주어야 replace  됨. 다시 download 받을 경우 ~/.keras file 삭제 필요

In [2]:
path_to_file = tf.keras.utils.get_file("shakespeare.txt", 
                         "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")

In [3]:
text = open(path_to_file, 'r').read()
len(text)

1115394

In [4]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### lookup table 작성

- 문자들을 숫자로 변환


- 두 개의 조회 테이블(lookup table) 작성  
    - character to index
    - index to character


- text 중에 포함된 character 들을 이용하여 charactet-to-index, index-to-character 변환 table 작성
    - 하나는 문자를 숫자에 매핑하고 다른 하나는 숫자를 문자에 매핑  
    - 문자를 0번 인덱스부터 고유 문자 길이까지 매핑

In [5]:
chars = sorted(set(text))
nb_chars = len(chars)

char2idx = dict((c, i) for i, c in enumerate(chars))
idx2char = chars

In [6]:
print("문자 갯수 : ", nb_chars)
print()
print("character to index lookup table : \n", char2idx)
print()
print("index to character lookup table : \n",idx2char)

문자 갯수 :  65

character to index lookup table : 
 {'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}

index to character lookup table : 
 ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


### 훈련 sample 과 target 만들기 

- input data 를 고정길이로 통일하기 위해 MAX_SEQ_LEN 을 입력의 최대 길이로 지정 (임의로 결정)   


- MAX_SEQ_LEN 개의 연속된 character sequence 를 input 으로 하고, 1 character shift 하여 이어지는 동일한 길이의 character sequence 를 label 로 만든다.  

- 따라서, MAX_SEQ_LEN + 1 길이의 덩어리(chunk)로 text 를 분할하여 input, label 분리 (tf.Dataset method 사용)
        ex) MAX_SEQ_LEN = 4 인 경우,
            "Hell"  --> "ello"
            

- 이를 손쉽게 하기 위해 tensorflow 가 제공하는 tf.data.Dataset.from_tensor_slices 함수를 사용하여 text vector 를 character index 의 stream 으로 변환  

In [7]:
MAX_SEQ_LEN = 100         # time step = 100, 단일 입력에 대해 원하는 문장의 최대 글자수
examples_per_epoch = len(text) // MAX_SEQ_LEN
print("한 epoch 당 처리할 전체 sample 수 :", examples_per_epoch)

한 epoch 당 처리할 전체 sample 수 : 11153


**char2index lookup table 을 이용하여 text 전체를 integer 로 변환**

In [8]:
text_as_int = np.array([char2idx[c] for c in text])
print(text_as_int)

[18 47 56 ... 45  8  0]


In [9]:
print("변환전 : {}, 변환후 : {}".format(text[:13], text_as_int[:13]))

변환전 : First Citizen, 변환후 : [18 47 56 57 58  1 15 47 58 47 64 43 52]


**훈련 sample 및 target 만들기**

- tf.data.Dataset.from_tensor_slices method $\rightarrow$ 주어진 tensor 들을 first dimension 을 따라 slice

In [10]:
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
char_dataset

<TensorSliceDataset shapes: (), types: tf.int32>

In [11]:
for idx in char_dataset.take(5):
    print(idx, idx.numpy())

tf.Tensor(18, shape=(), dtype=int32) 18
tf.Tensor(47, shape=(), dtype=int32) 47
tf.Tensor(56, shape=(), dtype=int32) 56
tf.Tensor(57, shape=(), dtype=int32) 57
tf.Tensor(58, shape=(), dtype=int32) 58


** tf.data 의 batch method 를 사용하면 개별 character 를 원하는 size 의 character sequence 로 쉽게 바꿀 수 있다**

- char_dataset 을 seq_len +1 길이의 character sequence 로 변환 (next character 를 예측하는 문제이므로 +1) 하여  
  [ : sequence]는 input 으로 사용하고 [1 : sequence+1] 은 label 로 사용  

In [12]:
sequences = char_dataset.batch(MAX_SEQ_LEN + 1, drop_remainder=True)

for idx in sequences.take(1):
    seq = [i for i in idx.numpy()]
    print(seq)
    print()
    print(repr(''.join(idx2char[i] for i in seq)))
    print()

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56, 1, 51, 43, 1, 57, 54, 43, 39, 49, 8, 0, 0, 13, 50, 50, 10, 0, 31, 54, 43, 39, 49, 6, 1, 57, 54, 43, 39, 49, 8, 0, 0, 18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 37, 53, 59, 1]

'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '



## dataset 생성 - input text, target text

- sequnces 를 입력 text 와 target text 로 분리하는 helper 함수를 작성하여 map 함수로 각 sequences 의 element 에 적용

In [13]:
def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

## Training / Target dataset 구성

In [14]:
sequences

<BatchDataset shapes: (101,), types: tf.int32>

In [15]:
dataset = sequences.map(split_input_target)

In [16]:
dataset

<MapDataset shapes: ((100,), (100,)), types: (tf.int32, tf.int32)>

In [17]:
for input_sample, target_sample in dataset.take(1):
    input_seq = [i for i in input_sample.numpy()]
    target_seq = [i for i in target_sample.numpy()]
    print("입력 data   = ", repr(''.join(idx2char[idx] for idx in input_seq)))
    print("출력 data   = ", repr(''.join(idx2char[idx] for idx in target_seq)))

입력 data   =  'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
출력 data   =  'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


- input_sample 의 각 인덱스는 하나의 타임 스텝(time step)으로 처리  

 
- time step 0의 입력으로 모델은 "F"의 인덱스를 받고 다음 문자로 "i"의 인덱스를 예측   


- 다음 타임 스텝에서도 같은 일을 하지만 RNN은 현재 입력 문자 외에 이전 타임 스텝의 컨텍스트(context)를 고려하여 예측  


- RNN 의 context 정보는 hidden state 를 뜻함

## Training batch 생성

- 텍스트를 다루기 쉬운 시퀀스로 분리하기 위해 tf.data를 사용하여 train batch 생성    


- 이 데이터를 모델에 넣기 전에 데이터를 섞은 후 배치를 만든다 (다음 character 를 예측하는 모델이고 임의의 batch size 로 잘랐으므로 shuffle 필요)  


- batch method 를 이용하여 train batch 생성

In [18]:
BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset

<BatchDataset shapes: ((64, 100), (64, 100)), types: (tf.int32, tf.int32)>

### Model 생성

<div>
<img src="https://tensorflow.org/tutorials/text/images/text_generation_training.png", width="400">
</div>

- tf.keras.Sequential model 사용  


- 3 개의 층을 사용하여 모델 정의  


    1. Embedding : 입력층, embedding_dim 차원 벡터에 각 문자의 정수 코드를 매핑하는 훈련가능한 검색 table
        - 각 character 에 대해 model 은 embedding 을 검색하고, embedding 을 입력으로 하여 GRU(LSTM) 를 1 개의 time step 으로 실행  

    2. LSTM (GRU) 를 이용한 RNN 층

    3. 완전연결층(Dense) 
        - logits 생성 : (64, 100, 65) - (batch size, MAX_SEQ_LEN, vocab_size)


- Dense layer 에서 생성된 크기가 vocab_size 인 logit 을 tf.random.categorical 에 적용하여 다음 문자의 확률분포 예측  


- stateful=True 로 지정 - 문장이 계속 연속되므로 이전 batch 의 hidden state 를 다음번 batch 의 초기 state 로 사용  

    $\rightarrow$ batch_input_shape 반드시 지정 ($t_i$ 의 initial state 가 $t_{i+bs}$ 로 연결되므로)


- 입력 batch shape : (64, 100)  
- 출력 batch shape : (64, 100, 65)

In [22]:
EMBEDDING_DIM = 256
RNN_UNITS = 1024
print(nb_chars)

65


**마지막 Dense layer 의 activation='linear' --> shape(100, 65)**

- last layer 의 activation 을 softmax 로 하는 것보다 계산상 안정적임

In [23]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, embedding_dim, batch_input_shape=[batch_size, None]),
        tf.keras.layers.LSTM(rnn_units, 
                             return_sequences=True, 
                             stateful=True,
                             recurrent_initializer='glorot_uniform'),
        tf.keras.layers.Dense(nb_chars)
    ])
    return model

model = build_model(vocab_size=nb_chars, embedding_dim=EMBEDDING_DIM, rnn_units=RNN_UNITS, batch_size=BATCH_SIZE)

In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (64, None, 256)           16640     
_________________________________________________________________
lstm (LSTM)                  (64, None, 1024)          5246976   
_________________________________________________________________
dense (Dense)                (64, None, 65)            66625     
Total params: 5,330,241
Trainable params: 5,330,241
Non-trainable params: 0
_________________________________________________________________


### 마지막 layer 에서 output sampling 요령

- distribution 의 argmax 를 하면 model 이 쉽게 loop 에 빠짐.  


- tf.random.categorical(logits, num_samples) 을 이용하여 categorical 분포로 부터 sample 추출  


    - logits: [batch_size, num_classes] 의 2-D Tensor. 각 slice [i, :] 는 각 class 의 normalize 되지 않은 log-probability 를 나타냄
    - num_samples: 각 row slice 에서 추출할  독립적 sample 수

In [26]:
# ex. 0 과 1 이 각 0.5 의 확률을 가진 [1, 5] shape 의 random 변수 생성
samples = tf.random.categorical(tf.math.log([[0.5, 0.5]]), num_samples=5)
samples

<tf.Tensor: id=653, shape=(1, 5), dtype=int64, numpy=array([[1, 1, 1, 0, 1]], dtype=int64)>

```
tf.random.categorical(example_batch_predictions[0], num_samples=1)
```
    --> MAX_SEQ_LEN=100 이므로, 100 개의 timestep 별로 65 개의 character 별 확률([100, 65) 로  1 개를 random sampling 
    --> [100, 1] output

## 모델 훈련
- 이 문제는 표준적인 분류 문제로 취급 가능  

- 이전 RNN 상태와 현재 타임 스텝(time step)의 입력으로 다음 문자의 클래스를 예측.

### 손실함수 지정

- 표준 tf.keras.losses.sparse_categorical_crossentropy 손실 함수는 prediction 의 마지막 dimension 전체에 적용되므로 이 경우에 잘 작동함 


    - categorical_crossentropy - one-hot-encoding  
        [1,0,0]
        [0,1,0]   
        [0,0,1]  
    - sparse_categorical_crossentropy - integer encoding
        1  
        2  
        3  


- 우리의 모델은 logit 을 출력으로 반환하므로 from_logits=True 로 setting 한다.
    --> softmax activation 보다 계산이 안정적 (Tensorflow document)

In [31]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(loss=loss, optimizer='adam')

## checkpoint 구성

- 훈련 중 checkpoint 저장

In [32]:
import os

checkpoint_dir = './training_checkpoints'
# 체크포인트 파일 이름
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True)

In [33]:
s = time.time()

history = model.fit(dataset, epochs=20, callbacks=[checkpoint_callback])

print(time.time() - s)

Epoch 1/20
172/172 [==============================] - 43s 247ms/step - loss: 2.6820
Epoch 2/20
172/172 [==============================] - 38s 224ms/step - loss: 2.0270
Epoch 3/20
172/172 [==============================] - 39s 224ms/step - loss: 1.7871
Epoch 4/20
172/172 [==============================] - 39s 225ms/step - loss: 1.6303
Epoch 5/20
172/172 [==============================] - 39s 225ms/step - loss: 1.5236
Epoch 6/20
172/172 [==============================] - 39s 225ms/step - loss: 1.4472
Epoch 7/20
172/172 [==============================] - 39s 225ms/step - loss: 1.3897
Epoch 8/20
172/172 [==============================] - 39s 225ms/step - loss: 1.3433
Epoch 9/20
172/172 [==============================] - 39s 226ms/step - loss: 1.3017
Epoch 10/20
172/172 [==============================] - 39s 225ms/step - loss: 1.2625
Epoch 11/20
172/172 [==============================] - 39s 226ms/step - loss: 1.2238
Epoch 12/20
172/172 [==============================] - 39s 226ms/step - lo

## 훈련된 model 을 이용한 text generation

- 예측 단계를 간단히 하기 위해 batch size = 1 로 변경한 새로운 model 을 생성하고, last checkpoint 의 model weight 를 load  


- model rebuild 이유:


    - 우리가 작성했던 model 은 작성 당시의 fixed batch size 만 받아 들이므로, 
    - 다른 batch size 에서도 수행되도록 하기 위해서는 model 을 batch_size 1 로 rebuild 하고 마지막 checkpoint 에서 저장한 model weight 를 restore

In [39]:
# batch size 1 의 new model
model = build_model(vocab_size=nb_chars, embedding_dim=EMBEDDING_DIM, rnn_units=RNN_UNITS, batch_size=1)
# weight load
model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
# batch size 1 로 rebuild
model.build(tf.TensorShape([1, None]))

model.summary()

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_2 (Embedding)      (1, None, 256)            16640     
_________________________________________________________________
lstm_2 (LSTM)                (1, None, 1024)           5246976   
_________________________________________________________________
dense_2 (Dense)              (1, None, 65)             66625     
Total params: 5,330,241
Trainable params: 5,330,241
Non-trainable params: 0
_________________________________________________________________


## 예측 loop 

다음 code block 에서 text 생성:

- start string 으로 시작, RNN state 초기화 및 생성할 characters 수 설정 

- start string 과 RNN state 를 이용하여 next character 의 prediction 분포를 가져옴  

- categorical 분포를 이용하여 예측된 character 의 index 계산. 예측된 character 를 model 의 next input 으로 사용  

- model 에서 반환된 RNN state 는 model 로 다시 되돌려져서 이제는 하나의 character 가 아닌 더 많은 문맥(context) 에 더해짐

- 다음 단어를 예측한 후 수정된 RNN state 가 다시 모델로 피드백되어 더 많은 context 를 얻으며 학습되는 방식임

![텍스트를 생성하기 위해 모델의 출력이 입력으로 피드백](https://tensorflow.org/tutorials/text/images/text_generation_sampling.png)

In [40]:
start_string = "ROMEO: "

num_generate = 1000   # 생성할 문자의 수

# starting_string 의 숫자화 (벡터화)
input_eval = [char2idx[c] for c in start_string]
input_eval = tf.expand_dims(input_eval, 0)
input_eval

<tf.Tensor: id=128640, shape=(1, 7), dtype=int32, numpy=array([[30, 27, 25, 17, 27, 10,  1]])>

stateful RNN 이므로 새로운 text generation 을 위해 reset_states() method 를 이용하여 initial state 초기화

In [41]:
text_generated = []

model.reset_states()

for i in range(num_generate):
    predictions = model(input_eval)    # (1, 1, 65)
    # batch dimension 제거
    predictions = tf.squeeze(predictions, 0)   # (1, 65)
    # argmax 아니고 분포에서 sampling
    predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy() # scalar
    
    if i == 1:
        print(predictions.shape)
        print(predictions)
        predicted = tf.random.categorical(predictions, num_samples=1)
        print(predicted)
        print(predicted[-1, 0])
    
    # 예측된 character 를 이전 은닉 상태와 함께 다음 입력으로 model 에 전달
    input_eval = tf.expand_dims([predicted_id], 0)   # (1, 1)
    text_generated.append(idx2char[predicted_id])

(1, 65)
tf.Tensor(
[[ -0.5322553    6.1339655    0.8115481   -6.649242    -6.610993
   -2.64268      3.9219952    0.62988514   1.4583844   -5.866259
   -1.6458609   -0.06571153  -1.9676152   -5.783365    -4.9901447
   -3.402115    -7.160193    -7.6968603   -4.9728365   -3.8077092
   -3.0965576   -2.3234618   -3.8976407   -5.1068063   -2.110152
   -3.659635    -6.562627    -0.766338    -4.0649805   -4.4772835
   -2.7808383   -5.030262    -5.5023103   -5.384765    -3.469045
   -3.35597     -7.06196     -5.481399    -6.2299576    3.5324223
   -3.9758165   -3.5033064   -0.6087679    6.2487564   -5.0910025
   -1.3298568   -2.2927332    7.1884837   -3.7678945   -6.8890004
   -1.3843963   -1.3454759   -2.040156     8.704313    -5.0595737
   -5.0149302    3.1958208    1.0728036  -11.6473875    4.99997
   -3.4941475    3.8018444   -5.432572     1.7615678   -6.6030917 ]], shape=(1, 65), dtype=float32)
tf.Tensor([[53]], shape=(1, 1), dtype=int64)
tf.Tensor(53, shape=(), dtype=int64)


In [42]:
print(start_string + ''.join(text_generated))

ROMEO: done fire less to him
I'll put you to me.

BAPTISTA:
What does tranioting hot?
Scravest that tale, and brushing crown'd body's pilies, in streetsmen in puke quar
To obey you poor sighs.

MIRANDA:
Good Peter o'er the meaning of the horse.
Who is't midnie?

BIONDELLO:
Upon my party, sir. Fare your welcome hords?

POMPEY:
O, sir, his panished scoop'd in lawful kindness, wherein
show them gown, let unto Bianca?

BAPP:
O, she books me, it shan? Tell me, thou hast manifest, yet ignoranter
Of my deeds, brother, only love, the horous subdue-citizens;
But in eventy tunes of rummiaRW:
Hadst your head for pray; not to be curst him!

VINCENTIO:
Take this feespord, who puts him where a winter? What o'clock,
That that with sweeds, you will arrive?

Masseager:
They, that there shall be the father find
And leam'd afarothed philosophy town?
Here lead this horse; for all hap shuntingagdrch and speed;
Too pregnier good, yet regard him to thy fire;
And not reserved us you to London to the crown.

P